# P05 — GPU offload spike (Phase 0)

The question this notebook answers: **can we run P05's OpenMP `target` output on a real
NVIDIA GPU from Colab/Kaggle?** Everything later in the evaluation depends on a yes.

Runtime → Change runtime type → **T4 GPU** (Colab) or **GPU P100/T4** (Kaggle), then
Run all.

* **Path A** — LLVM clang with libomptarget (apt.llvm.org). Preferred: same compiler
  family the tool is built on.
* **Path B** — NVIDIA HPC SDK `nvc -mp=gpu`. Fallback if Path A can't offload.

A path counts as working only if the **probe** reports that the target region ran on a
non-host device under `OMP_TARGET_OFFLOAD=MANDATORY` (which aborts instead of silently
falling back to the CPU). Then the real tool output `benchmarks/compute_heavy.omp.c` is
run on the GPU and checked against a sequential host baseline.

The last cell prints a JSON summary — paste it back to Claude / commit it under
`eval/colab/results/`.

In [ ]:
%%bash
nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total --format=csv
nvcc --version 2>/dev/null | tail -2 || echo "no nvcc on PATH"
ls -d /usr/local/cuda* 2>/dev/null
cat /etc/os-release | grep PRETTY

In [ ]:
import subprocess, json, os, pathlib
gpu = subprocess.run(["nvidia-smi","--query-gpu=name,compute_cap","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().splitlines()[0]
GPU_NAME, CC = [s.strip() for s in gpu.split(",")]
SM = "sm_" + CC.replace(".", "")
os.environ["P05_SM"] = SM
RESULTS = {"gpu": GPU_NAME, "sm": SM}
print(GPU_NAME, SM)

In [ ]:
%%bash
set -e
cd /content 2>/dev/null || cd /kaggle/working
rm -rf p05 && git clone -q --depth 1 https://github.com/adhithyaragavan/gpgpu-autoparallel.git p05
rm -rf spike && mkdir spike && cd spike
# Probe: succeeds only if the target region really executes on a device.
cat > probe.c <<'EOF'
#include <omp.h>
#include <stdio.h>
int main(void) {
  int on_host = 1, ndev = omp_get_num_devices();
  #pragma omp target map(from: on_host)
  { on_host = omp_is_initial_device(); }
  printf("num_devices=%d ran_on_host=%d\n", ndev, on_host);
  return (ndev > 0 && !on_host) ? 0 : 1;
}
EOF
# Sequential baseline of the real benchmark (same source, pragmas ignored).
# Note: the baseline is gcc -O2, a different compiler from the GPU build.
gcc -O2 -o seq ../p05/benchmarks/compute_heavy.c
./seq seq.bin
pwd

## Path A — LLVM clang + libomptarget (apt.llvm.org)

In [ ]:
%%bash
cd /tmp
wget -q https://apt.llvm.org/llvm.sh && chmod +x llvm.sh
./llvm.sh > llvm_install.log 2>&1 || { tail -20 llvm_install.log; exit 1; }
V=$(ls -d /usr/lib/llvm-* | sed 's/.*llvm-//' | sort -n | tail -1)
apt-get install -y -qq libomp-$V-dev clang-tools-$V 2>&1 | tail -5 || echo "apt install failed"
echo "LLVM version: $V"
echo $V > /tmp/llvm_ver
# What offload pieces did we get?
ls /usr/lib/llvm-$V/lib | grep -i -E 'omptarget|offload|nvptx' || echo "NO offload runtime libraries found"
echo "--- device runtime dir:"; ls /usr/lib/llvm-$V/lib/nvptx64-nvidia-cuda 2>&1 | head
echo "--- offload driver tools:"; ls /usr/lib/llvm-$V/bin | grep -E "linker-wrapper|offload-packager" || echo "missing"

In [ ]:
%%bash
cd /content/spike 2>/dev/null || cd /kaggle/working/spike
V=$(cat /tmp/llvm_ver); CL=/usr/lib/llvm-$V/bin/clang
FLAGS="-O2 -fopenmp --offload-arch=$P05_SM --cuda-path=/usr/local/cuda -Wno-unknown-cuda-version"
export LD_LIBRARY_PATH=/usr/lib/llvm-$V/lib:$LD_LIBRARY_PATH
if ! $CL $FLAGS -o probe_clang probe.c 2> pathA.err; then
  echo "PATH_A_PROBE=FAIL stage=compile" | tee pathA.status
elif ! OMP_TARGET_OFFLOAD=MANDATORY ./probe_clang 2>> pathA.err; then
  echo "PATH_A_PROBE=FAIL stage=run" | tee pathA.status
else
  echo "PATH_A_PROBE=OK" | tee pathA.status
fi
[ -s pathA.err ] && { echo "--- stderr (also saved in pathA.err):"; tail -20 pathA.err; }
if grep -q OK pathA.status; then
  $CL $FLAGS -o heavy_clang ../p05/benchmarks/compute_heavy.omp.c
  # Proof of a real kernel launch + transfers, from libomptarget itself (to a log,
  # not through `head`, which could SIGPIPE the program before it writes its dump):
  OMP_TARGET_OFFLOAD=MANDATORY LIBOMPTARGET_INFO=4 ./heavy_clang heavy_clang.bin > heavy_clang.log 2>&1
  head -25 heavy_clang.log
fi

## Path B — NVIDIA HPC SDK (only run if Path A printed `PATH_A_PROBE=FAIL`)

Multi-GB download; takes several minutes.

In [ ]:
%%bash
cd /content/spike 2>/dev/null || cd /kaggle/working/spike
if grep -q OK pathA.status 2>/dev/null; then echo "Path A works - skipping Path B"; exit 0; fi
curl -s https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK \
  | gpg --dearmor --yes -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' \
  | tee /etc/apt/sources.list.d/nvhpc.list > /dev/null
apt-get update -qq
PKG=$(apt-cache search --names-only '^nvhpc-[0-9]+-[0-9]+$' | awk '{print $1}' | sort -V | tail -1)
echo "Installing $PKG"; apt-get install -y -qq $PKG > /tmp/nvhpc.log 2>&1 || tail -20 /tmp/nvhpc.log
NVC=$(ls -d /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc | sort -V | tail -1)
CCN=${P05_SM#sm_}
status=FAIL
$NVC -O2 -mp=gpu -gpu=cc$CCN -o probe_nvc probe.c && OMP_TARGET_OFFLOAD=MANDATORY ./probe_nvc && status=OK
echo "PATH_B_PROBE=$status" | tee pathB.status
if [ $status = OK ]; then
  $NVC -O2 -mp=gpu -gpu=cc$CCN -Minfo=mp -o heavy_nvc ../p05/benchmarks/compute_heavy.omp.c
  OMP_TARGET_OFFLOAD=MANDATORY NVCOMPILER_ACC_NOTIFY=3 ./heavy_nvc heavy_nvc.bin > heavy_nvc.log 2>&1
  head -25 heavy_nvc.log
fi

## Correctness + timing on the GPU, and the summary

In [ ]:
import numpy as np, re, statistics
os.chdir("/content/spike" if os.path.isdir("/content/spike") else "/kaggle/working/spike")
seq = np.fromfile("seq.bin", dtype=np.float64)

def timing(cmd, reps=7):
    ts = []
    for _ in range(reps):
        r = subprocess.run(cmd, capture_output=True, text=True,
                           env={**os.environ, "OMP_TARGET_OFFLOAD": "MANDATORY"})
        m = re.search(r"TIMING \S+ ([\d.]+)", r.stderr)
        if r.returncode != 0 or not m:
            return None, r.returncode
        ts.append(float(m.group(1)))
    return min(ts), 0

RESULTS["seq_us"] = timing(["./seq"])[0]
RESULTS["seq_compiler"] = "gcc -O2"
RESULTS["timing_note"] = "min of 7 fresh processes: every run includes CUDA context creation; cold-start data, not a speedup"
for path, binary in [("A_clang", "heavy_clang"), ("B_nvc", "heavy_nvc")]:
    if not os.path.exists(binary):
        RESULTS[path] = "not built"; continue
    t, rc = timing(["./" + binary])
    out = np.fromfile(binary + ".bin", dtype=np.float64) if os.path.exists(binary + ".bin") else None
    entry = {"gpu_min_us_of_7": t, "exit": rc}
    if out is not None and out.shape == seq.shape:
        entry["bitwise_equal"] = int(np.sum(out.view(np.uint64) == seq.view(np.uint64)))
        entry["max_rel_err"] = float(np.max(np.abs(out - seq) / np.maximum(np.abs(seq), 1e-300)))
        entry["n"] = int(seq.size)
    RESULTS[path] = entry

for f in ["pathA.status", "pathB.status"]:
    if os.path.exists(f): RESULTS[f] = open(f).read().strip()
if os.path.exists("pathA.err"):
    RESULTS["pathA_stderr_tail"] = open("pathA.err").read().strip().splitlines()[-8:]
print(json.dumps(RESULTS, indent=2))

**Reading the result.** `bitwise_equal < n` is expected and not a bug by itself: the GPU
contracts `r*x + c` into FMA instructions, which round differently from the host's
separate multiply and add. A `max_rel_err` around 1e-15 means it's correct. Large errors
are real failures. Timing here includes the first-touch CUDA context creation, so
expect it to look slow — separating that cost out is the point of the Phase 2
calibration benchmarks.